# PDE Option Pricer — Walkthrough

This notebook demonstrates the four PDE solvers in the `engine/` module:
1. **Crank-Nicolson** — European & American pricing in log-space
2. **Barrier + Local Vol** — Implicit FD with σ(S) = σ_atm·(S/K)^{-α}
3. **Free Boundary** — Extraction of the early-exercise boundary S*(t)
4. **PSOR + Dividends** — Projected SOR with discrete dividend adjustment

In [ ]:
import numpy as np
import plotly.graph_objects as go
from engine import (
    GridConfig, bs_price, bs_delta, bs_gamma, bs_vega,
    compute_price_surface,
    price_european_american,
    price_barrier_local_vol,
    extract_free_boundary,
    price_american_dividends_psor,
)

## 1. European & American — Crank-Nicolson

The Black-Scholes PDE in log-space x = ln(S):

$$\frac{\partial V}{\partial t} + \frac{\sigma^2}{2}\frac{\partial^2 V}{\partial x^2} + \left(r - \frac{\sigma^2}{2}\right)\frac{\partial V}{\partial x} - rV = 0$$

Discretised with the Crank-Nicolson (θ = 0.5) scheme for second-order accuracy in both time and space.

In [ ]:
K, T, r, sigma = 100.0, 1.0, 0.05, 0.20
grid = GridConfig(N=500, M=500)

eu_call = price_european_american(K, T, r, sigma, 'call', 'european', grid)
am_put  = price_european_american(K, T, r, sigma, 'put', 'american', grid)
bs_ref  = bs_price(K, K, T, r, sigma, 'call')

print(f'European Call (PDE): {eu_call.price:.5f}')
print(f'BS Analytical:      {float(bs_ref):.5f}')
print(f'Error:              {abs(eu_call.price - float(bs_ref)):.5f}')
print(f'American Put (PDE): {am_put.price:.5f}')

In [ ]:
S = eu_call.S_grid
mask = (S > 50) & (S < 150)
bs_curve = bs_price(S, K, T, r, sigma, 'call')
payoff = np.maximum(S - K, 0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=S[mask], y=eu_call.V_grid[mask], name='PDE'))
fig.add_trace(go.Scatter(x=S[mask], y=bs_curve[mask], name='BS', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=S[mask], y=payoff[mask], name='Payoff', line=dict(color='grey')))
fig.update_layout(title='European Call — PDE vs BS', xaxis_title='S', yaxis_title='V',
                  template='plotly_white', height=400)
fig.show()

## 2. Barrier Option with Local Volatility

An up-and-out call with local vol σ(S) = σ_atm · (S/K)^{-α} solved via implicit finite differences on a uniform S-grid.

In [ ]:
bar = price_barrier_local_vol(K=100, T=1.0, r=0.05, sigma_atm=0.20, alpha=0.4, B=130.0, grid=grid)
print(f'Barrier Call price at S=100: {bar.price:.4f}')

mask_b = bar.S_grid < 135
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=bar.S_grid[mask_b], y=bar.V_grid[mask_b], name='Barrier Call'))
fig2.add_vline(x=130, line_dash='dash', line_color='red', annotation_text='B=130')
fig2.update_layout(title='Up-and-Out Call', xaxis_title='S', yaxis_title='V',
                  template='plotly_white', height=400)
fig2.show()

## 3. Free-Boundary Extraction

For an American put, we extract the optimal exercise boundary S*(t) by identifying where V = max(K - S, 0) at each time step.

In [ ]:
fb = extract_free_boundary(K=100, T=1.0, r=0.05, sigma=0.20, option_type='put',
                           grid=GridConfig(N=200, M=200))

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=fb.time_grid, y=fb.boundary, name='S*(t)'))
fig3.add_hline(y=100, line_dash='dash', line_color='grey', annotation_text='K=100')
fig3.update_layout(title='American Put — Exercise Boundary', xaxis_title='t', yaxis_title='S*',
                  template='plotly_white', height=400)
fig3.show()

## 4. PSOR with Discrete Dividends

Projected SOR iteratively solves the complementarity problem at each time step. A discrete cash dividend is handled by interpolating V(S − D) onto the grid at the ex-dividend date.

In [ ]:
psor = price_american_dividends_psor(
    K=100, T=1.0, r=0.05, sigma=0.25, option_type='call',
    div_amount=5.0, div_time=0.48, omega=1.5,
    grid=GridConfig(N=150, M=250),
)
print(f'PSOR Price: {psor.price:.4f}  (iterations: {psor.iterations})')

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=psor.time_grid, y=psor.free_boundary, name='S*(t)'))
fig4.add_vline(x=0.48, line_dash='dash', line_color='red', annotation_text='Ex-div')
fig4.update_layout(title='PSOR Free Boundary with Dividend', xaxis_title='t', yaxis_title='S*',
                  template='plotly_white', height=400)
fig4.show()

## 5. 3D Price Surface

Black-Scholes price over the (S, T) grid — useful for intuition about time decay and moneyness.

In [ ]:
surf = compute_price_surface(K=100, r=0.05, sigma=0.25, option_type='put',
                             S_min=50, S_max=150, n_S=100, T_min=0.01, T_max=1.0, n_T=100)

fig5 = go.Figure(data=[go.Surface(x=surf.T_values, y=surf.S_values, z=surf.V_surface.T,
                                   colorscale='Viridis')])
fig5.update_layout(scene=dict(xaxis_title='T', yaxis_title='S', zaxis_title='V'),
                   title='BS Put Price Surface', template='plotly_white', height=500)
fig5.show()